<a href="https://colab.research.google.com/github/yyyanano/document-analyzer/blob/main/document_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Colab Document OCR с моделью Donut
# Анализирует изображения и PDF, извлекает поля и генерирует JSON

!apt-get update --fix-missing -y
!apt-get install -y poppler-utils

!pip install transformers donut-python pillow matplotlib pdf2image PyMuPDF --quiet

import json
import torch
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os
from google.colab import files
from datetime import datetime
import fitz
from io import BytesIO

print("Библиотеки загружены")
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    torch.set_num_threads(2)
    print("Работаем на CPU. Модель может загружаться дольше.")

def load_model():
    """Загружает предобученную модель Donut для анализа документов"""

    model_name = "AndreasPiper/donut-base-funsd"
    print(f"Загрузка модели {model_name}...")
    print("Это займет 5-10 минут на CPU (или 1-2 минуты на GPU)")

    try:
        processor = DonutProcessor.from_pretrained(model_name)
        model = VisionEncoderDecoderModel.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True
        )

        if torch.cuda.is_available():
            model = model.to("cuda")
            print("✓ Модель загружена на GPU")
        else:
            print("✓ Модель загружена на CPU")

        model.eval()
        return processor, model

    except Exception as e:
        print(f"✗ Ошибка загрузки модели: {e}")
        raise

processor, model = load_model()

def convert_pdf_to_image_pymupdf(pdf_bytes):
    """Конвертирует первую страницу PDF в RGB-изображение"""

    try:
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        page = doc[0]

        zoom = 2.0  # Увеличиваем качество в 2 раза
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat)

        img_data = pix.tobytes("png")
        image = Image.open(BytesIO(img_data)).convert("RGB")

        doc.close()
        print("✓ PDF успешно конвертирован (PyMuPDF)")
        return image

    except Exception as e:
        print(f"✗ Ошибка конвертации PDF: {e}")
        return None

def load_document(file_bytes, file_name):
    """Загружает изображение или PDF-файл"""

    ext = file_name.lower().split('.')[-1]

    if ext in ['jpg', 'jpeg', 'png', 'bmp', 'tiff', 'webp']:
        print("✓ Загружено изображение")
        return Image.open(BytesIO(file_bytes)).convert("RGB")

    elif ext == 'pdf':
        print("⏳ Конвертация PDF в изображение...")
        image = convert_pdf_to_image_pymupdf(file_bytes)
        if image is None:
            raise ValueError("Не удалось конвертировать PDF")
        return image

    else:
        raise ValueError(
            f"✗ Неподдерживаемый формат: {ext}\n"
            f"  Поддерживаются: JPG, PNG, JPEG, PDF"
        )

def predict_document(image, processor, model):
    """Анализирует документ, возвращает JSON с полями и bboxes"""

    if not isinstance(image, Image.Image):
        image = Image.open(image).convert("RGB")

    # Оптимизируем размер
    max_size = 1024
    if max(image.size) > max_size:
        image.thumbnail((max_size, max_size))
        print(f"  (изображение уменьшено до {image.size})")

    # Подготовка пикселей
    pixel_values = processor(image, return_tensors="pt").pixel_values

    if torch.cuda.is_available():
        pixel_values = pixel_values.to("cuda")

    # Генерация предсказания
    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            max_length=512,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            num_beams=1,
            return_dict_in_generate=True,
        )

    generated_text = processor.batch_decode(
        outputs.sequences,
        skip_special_tokens=True
    )[0]

    print(f"Сырой вывод модели: {generated_text}")

    # Парсим JSON из вывода
    try:
        start = generated_text.find('{')
        end = generated_text.rfind('}') + 1

        if start != -1 and end != -1:
            json_str = generated_text[start:end]
            result = json.loads(json_str)
        else:
            result = {"error": "Не найден JSON", "raw": generated_text}

    except json.JSONDecodeError as e:
        result = {"error": f"Ошибка парсинга JSON: {e}", "raw": generated_text}

    return result, image

def visualize_result(image, result, title="Результат анализа"):
    """Рисует обнаруженные поля на изображении (bounding boxes)"""

    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(image)
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold')

    # Рисуем bounding boxes
    if "fields" in result:
        for field in result["fields"]:
            if "bbox" in field:
                x0, y0, x1, y1 = field["bbox"]
                ftype = field.get("type", "unknown")

                # Выбираем цвет по типу поля
                if ftype == "question":
                    color = 'red'
                elif ftype == "answer":
                    color = 'blue'
                else:
                    color = 'green'

                # Рисуем прямоугольник
                rect = patches.Rectangle(
                    (x0, y0), x1 - x0, y1 - y0,
                    linewidth=2, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)

                # Подпись типа
                ax.text(x0, y0 - 5, ftype,
                       fontsize=9, color='black',
                       bbox=dict(boxstyle='round',
                                facecolor='white', alpha=0.7))

    plt.tight_layout()
    plt.show()
    return fig

print("\n" + "="*70)
print("ЗАГРУЗКА ДОКУМЕНТА")
print("Поддерживаются форматы: JPG, PNG, JPEG, PDF")
print("="*70)

uploaded = files.upload()

if not uploaded:
    print("✗ Файл не загружен")
else:
    file_name = list(uploaded.keys())[0]
    file_bytes = uploaded[file_name]
    print(f"✓ Загружен файл: {file_name}")

    print("\n" + "="*70)
    print("ОБРАБОТКА ДОКУМЕНТА")
    print("="*70)

    try:
        image = load_document(file_bytes, file_name)
        print("✓ Документ загружен успешно")

        print("⏳ Анализ документа (это может занять минуту)...")
        result, image = predict_document(image, processor, model)

        print("\n" + "="*70)
        print("РЕЗУЛЬТАТ АНАЛИЗА (JSON)")
        print("="*70)
        print(json.dumps(result, ensure_ascii=False, indent=2))

        # Статистика
        if "fields" in result:
            questions = sum(1 for f in result["fields"] if f.get("type") == "question")
            answers = sum(1 for f in result["fields"] if f.get("type") == "answer")
            links = len(result.get("links", []))

            print("\n" + "-"*70)
            print("СТАТИСТИКА:")
            print(f"  • Вопросов: {questions}")
            print(f"  • Ответов: {answers}")
            print(f"  • Связей: {links}")
            print("-"*70)

        print("\nВИЗУАЛИЗАЦИЯ:")
        print("  Красный = вопрос, Синий = ответ, Зелёный = другой тип")
        visualize_result(image, result, f"Анализ документа: {file_name}")

        # Сохранение результатов
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        json_filename = f"result_{timestamp}.json"
        with open(json_filename, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"\n✓ JSON сохранён: {json_filename}")

        # Сохранение визуализации
        vis_filename = f"visualization_{timestamp}.png"
        fig, ax = plt.subplots(figsize=(14, 10))
        ax.imshow(image)
        ax.axis('off')

        if "fields" in result:
            for field in result["fields"]:
                if "bbox" in field:
                    x0, y0, x1, y1 = field["bbox"]
                    ftype = field.get("type", "unknown")
                    color = 'red' if ftype == "question" else 'blue' if ftype == "answer" else 'green'

                    rect = patches.Rectangle(
                        (x0, y0), x1 - x0, y1 - y0,
                        linewidth=2, edgecolor=color, facecolor='none'
                    )
                    ax.add_patch(rect)
                    ax.text(x0, y0 - 5, ftype,
                           fontsize=9, color='black',
                           bbox=dict(boxstyle='round',
                                    facecolor='white', alpha=0.7))

        plt.tight_layout()
        plt.savefig(vis_filename, dpi=150, bbox_inches='tight')
        print(f"✓ Визуализация сохранена: {vis_filename}")

        print("\n" + "="*70)
        print("СКАЧИВАНИЕ РЕЗУЛЬТАТОВ")
        print("="*70)
        files.download(json_filename)
        files.download(vis_filename)
        print("✓ Файлы готовы к скачиванию")

        print("\n✓ ГОТОВО!")

    except Exception as e:
        print(f"\n✗ ОШИБКА: {e}")
        import traceback
        traceback.print_exc()
        print("\nПроверьте, что файл является изображением или PDF с текстом")